[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/10-pretraining.ipynb)

# Module 10: Pretraining

Companion notebook to [Module 10](https://transformers-and-math.drmhse.com/modules/10-pretraining) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math, random
import numpy as np
import torch
import matplotlib.pyplot as plt

## Part 1 — Near-duplicate detection with MinHash (section 10.1)

In [ ]:
def shingles(text, k=3):
    w = text.lower().split()
    return {' '.join(w[i:i + k]) for i in range(len(w) - k + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b)

def minhash(s, n=128):
    return [min(hash((seed, x)) for x in s) for seed in range(n)]

docs = {
    'original': 'the committee met on tuesday to review the budget for the coming year and agreed to fund two new libraries in the north of the city',
    'edited': 'the committee met on tuesday to review the budget for next year and agreed to fund two new libraries in the north of the city',
    'unrelated': 'heavy rain is expected across the region this weekend so residents are advised to secure loose objects and avoid low roads',
}
A = shingles(docs['original'])
for name in ('edited', 'unrelated'):
    B = shingles(docs[name])
    est = np.mean(np.array(minhash(A)) == np.array(minhash(B)))
    print(f'{name:9}  exact Jaccard {jaccard(A, B):.3f}   MinHash estimate {est:.3f}')

## Part 2 — AdamW, written out (section 10.2)
One update by hand, checked against `torch.optim.AdamW`.

In [ ]:
torch.manual_seed(0)
w = torch.randn(5, requires_grad=True)
opt = torch.optim.AdamW([w], lr=0.1, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1)
w0 = w.detach().clone()
loss = (w ** 2).sum(); loss.backward()
g = w.grad.clone()
m, v = 0.1 * g, 0.05 * g ** 2                       # first step: (1 − β) times the gradient
m_hat, v_hat = m / (1 - 0.9), v / (1 - 0.95)
manual = w0 - 0.1 * (m_hat / (v_hat.sqrt() + 1e-8) + 0.1 * w0)
opt.step()
print('matches torch.optim.AdamW:', torch.allclose(manual, w.detach(), atol=1e-6))

## Part 3 — Warmup and cosine decay (section 10.3)

In [ ]:
def lr_at(step, peak=3e-4, warmup=1000, total=20000, floor=0.1):
    if step < warmup:
        return peak * step / warmup
    t = (step - warmup) / (total - warmup)
    return peak * (floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * t)))

steps = range(0, 20000, 50)
plt.plot(steps, [lr_at(s) for s in steps]); plt.xlabel('step'); plt.ylabel('learning rate'); plt.show()

## Part 4 — Compute and the Chinchilla rule (section 10.5)

In [ ]:
def compute(N, D):
    return 6 * N * D

print(f'8B params × 15T tokens: {compute(8e9, 15e12):.2e} FLOPs')
C = 5.76e23
N = math.sqrt(C / 120)                               # D = 20N  →  C = 120 N²
print(f'compute-optimal at C = {C:.2e}: N ≈ {N / 1e9:.0f}B parameters, D ≈ {20 * N / 1e12:.1f}T tokens')

E, A_, B_, a, b = 1.82, 482.01, 2085.43, 0.3478, 0.3658   # refitted constants (Besiroglu et al., 2024)
loss = lambda N, D: E + A_ / N ** a + B_ / D ** b
Ns = np.logspace(9, 12, 200)
plt.semilogx(Ns, [loss(n, C / (6 * n)) for n in Ns])
plt.xlabel('parameters N (compute held fixed)'); plt.ylabel('predicted loss'); plt.show()

## Your turn
1. How many GB does AdamW's training state take for a 7B model at 16 bytes per parameter?
2. Change the shingle length to 5 in Part 1. Why does the edited copy's similarity drop?